In [1]:
import csv
import time
from Bio import Entrez
from io import StringIO

def get_all_srr_from_biosample(biosample_id, email="your_email@example.com", max_retries=3):
    Entrez.email = email
    sra_uids = []

    for attempt in range(max_retries):
        try:
            handle = Entrez.esearch(db="sra", term=biosample_id)
            record = Entrez.read(handle)
            handle.close()
            sra_uids = record.get("IdList", [])
            break
        except:
            time.sleep(2)
    else:
        return 'Failed'

    if not sra_uids:
        return []

    runinfo_data = None
    for attempt in range(max_retries):
        try:
            handle = Entrez.efetch(
                db="sra",
                id=",".join(sra_uids),
                rettype="runinfo",
                retmode="text"
            )
            runinfo_data = handle.read()
            handle.close()
            break
        
        except:
            time.sleep(2)
    else:
        return 'Failed'

    try:
        if isinstance(runinfo_data, bytes):
            runinfo_data = runinfo_data.decode("utf-8")

        result = []
        reader = csv.DictReader(StringIO(runinfo_data))

        for row in reader:
            result.append({
                "run_id": row.get("Run", "").strip(),
                "platform": row.get("Platform", "").strip(),
                "layout": row.get("LibraryLayout", "").strip()
            })
        return result
    except:
        return 'data_error'

In [2]:
import os
import jsonlines
import pandas as pd

os.chdir('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data')
acc_info = []
with open('assembly_data_report.jsonl', 'r') as file:
    for line in jsonlines.Reader(file):
        temp_dict = {}
        temp_dict['accession'] = line['accession']
        temp_dict['organismName'] = line['organism']['organismName']
        temp_dict['taxId'] = line['organism']['taxId']
        temp_dict['biosample'] = line['assemblyInfo']['biosample']['accession']
        acc_info.append(pd.DataFrame([temp_dict]))

acc_info = pd.concat(acc_info, ignore_index=True)

In [3]:
from tqdm import tqdm
import threading
import multiprocessing

acc_info["srr_list"] = ""

def run_srr_fetch(i, biosample_id, que):
    timeout_sec = 60
    result = None

    def task():
        nonlocal result
        try:
            result = get_all_srr_from_biosample(biosample_id, email="your_email@example.com", max_retries=3)
        except:
            result = 'Failed'

    t = threading.Thread(target=task)
    t.daemon = True
    t.start()
    t.join(timeout=timeout_sec)

    if t.is_alive():
        que.put([i, ''])
    else:
        que.put([i, str(result) if result != 'Failed' else 'Failed'])

index = 0
while 'Failed' in acc_info["srr_list"].to_list() or '' in acc_info["srr_list"].to_list():
    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 64
    tot = 0
    pool = multiprocessing.Pool(par)
    
    for i, row in acc_info.iterrows():
        if acc_info.loc[i, 'srr_list'] == 'Failed' or acc_info.loc[i, 'srr_list'] == '':
            pool.apply_async(run_srr_fetch, (i, row['biosample'], que))
            tot += 1
        
    pool.close()

    count = 0
    index += 1
    with tqdm(total = tot, desc=f'srr_list_fetch ({index})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            if not que.empty():
                i, value = que.get(True)
                acc_info.loc[i, 'srr_list'] = value
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
                

srr_list_fetch (1): 100%|█████████████████████████████████████| 51.8k/51.8k [2:36:28<00:00, 5.51B/s]
srr_list_fetch (2): 100%|█████████████████████████████████████| 33.3k/33.3k [1:46:07<00:00, 5.23B/s]
srr_list_fetch (3): 100%|█████████████████████████████████████| 21.8k/21.8k [1:08:41<00:00, 5.29B/s]
srr_list_fetch (4): 100%|███████████████████████████████████████| 15.2k/15.2k [52:31<00:00, 4.82B/s]
srr_list_fetch (5): 100%|███████████████████████████████████████| 10.6k/10.6k [38:26<00:00, 4.58B/s]
srr_list_fetch (6): 100%|███████████████████████████████████████| 7.49k/7.49k [28:20<00:00, 4.40B/s]
srr_list_fetch (7): 100%|███████████████████████████████████████| 5.38k/5.38k [21:36<00:00, 4.15B/s]
srr_list_fetch (8): 100%|███████████████████████████████████████| 3.90k/3.90k [16:07<00:00, 4.03B/s]
srr_list_fetch (9): 100%|███████████████████████████████████████| 2.83k/2.83k [12:08<00:00, 3.88B/s]
srr_list_fetch (10): 100%|██████████████████████████████████████| 2.05k/2.05k [08:43<00:00,

In [4]:
os.chdir('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/')
acc_info.to_csv('biosample_info.tsv', sep='\t', index=False)